### Toy Example for plut-mab

```bash
pip install pluto
```

In [1]:
pip install pluto

/home/jorge/TFG/pluto/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import random

from pluto.eval.datasets import SquadV2Loader
from pluto.rag.config import RAGConfig
from pluto.rag.utils import SHORT_ANSWER_PROMPT
from pluto.rl.rl import DatasetRewardScorer, RagAction
from pluto.clustering.embeders import QuestionEmbedder
from pluto.clustering.clusters import AgglomerativeQuestionClusterer
from pluto.selection.config_selector import MABSelector
from pluto.selection.selection_pipeline import ClusterSelectionPipeline

# Load a random SQuAD v2 article
loader = SquadV2Loader()
title, items, long_context = loader.load_random_article()
print(f"Article: {title} ({len(items)} questions)")

# RAG config with the article as context
config = RAGConfig(
    texts=[long_context],
    llm_model_name="llama3.1:8b",
    llm_provider="ollama",
    custom_prompt_template=SHORT_ANSWER_PROMPT,
)
action_space = [
    RagAction(chunk_size=k, llm_temperature=t)
    for k in [100, 300, 500, 700, 1000]
    for t in [0.0, 0.3, 0.7, 1.0]
]

# UCB bandit selector
selector = MABSelector(
    config_template=config,
    reward_scorer=DatasetRewardScorer(),
    action_space=action_space,
)

# Cluster questions and optimize one configuration per cluster
embedder = QuestionEmbedder("sentence-transformers/all-mpnet-base-v2")
clusterer = AgglomerativeQuestionClusterer(embedder=embedder)
pipeline = ClusterSelectionPipeline(clusterer=clusterer, selector=selector)
result = pipeline.run(items=items, min_clusters=2, max_clusters=5, trials=100)

Using the latest cached version of the dataset since squad_v2 couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'squad_v2' at /home/jorge/.cache/huggingface/datasets/squad_v2/squad_v2/0.0.0/3ffb306f725f7d2ce8394bc1873b24868140c412 (last modified on Thu Nov 20 04:13:47 2025).
